In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalogo = "medalhao"
silver_db_name = "silver"

#data frame das tabelas do bronze
df_customers = spark.table(f"{catalogo}.bronze.tb_customers")
df_orders = spark.table(f"{catalogo}.bronze.tb_orders")
df_order_items = spark.table(f"{catalogo}.bronze.tb_order_items")
df_geolocalizacao = spark.table(f"{catalogo}.bronze.tb_geolocalizacao")
df_items = spark.table(f"{catalogo}.bronze.tb_items")
df_order_payments = spark.table(f"{catalogo}.bronze.tb_order_payments")
df_order_reviews = spark.table(f"{catalogo}.bronze.tb_order_reviews")
df_products = spark.table(f"{catalogo}.bronze.tb_products")
df_sellers = spark.table(f"{catalogo}.bronze.tb_sellers")
df_product_category_name_translation = spark.table(f"{catalogo}.bronze.tb_product_category_name_translation")
df_cotacao = spark.table(f"{catalogo}.bronze.tb_cotacao_dolar")





In [0]:
dim_consumidores = (df_customers
                          .withColumnRenamed("customer_id", "id_consumidor")
                          .withColumnRenamed("customer_unique_id", "id_consumidor_unico")
                          .withColumnRenamed("customer_gender", "cosumidor_genero")
                          .withColumnRenamed("customer_birth_date", "cosumidor_data_nascimento")
                          .withColumnRenamed("customer_age", "consumidor_idade")
                          .withColumnRenamed("customer_zip_code_prefix", "prefixo_cep")
                          .withColumnRenamed("customer_city", "cidade")
                          .withColumnRenamed("customer_state", "estado")
                          .withColumnRenamed("customer_name", "nome_consumidor"))

dim_consumidores = (dim_consumidores.withColumn("cidade", F.upper(F.col("cidade"))) #aplicando a regra de negócio
                           .withColumn("estado", F.upper(F.col("estado"))))

janela_deduplicacao_consumidores = Window.partitionBy("id_consumidor").orderBy(F.col("timestamp_ingestion")) #tirando os ID duplos com ordem decrescente de ingestao(recente)

dim_consumidores = (dim_consumidores
                           .withColumn("rn", F.rank().over(janela_deduplicacao_consumidores))
                           .filter(F.col("rn") == 1)
                           .drop("rn"))

dim_consumidores = dim_consumidores.withColumn("prefixo_cep", F.col("prefixo_cep").cast("string")) #deixar cep como string pois se tiver 0 a esquerda como inteiro é ignorado

(dim_consumidores
 .write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(f"{catalogo}.{silver_db_name}.dim_consumidores"))





In [0]:
dim_consumidores.display()

In [0]:
fat_pedidos = (df_orders.withColumnRenamed("order_id", "id_pedido")
                      .withColumnRenamed("customer_id", "id_consumidor")
                      .withColumnRenamed("order_purchase_timestamp", "data_compra")
                      .withColumnRenamed("order_approved_at", "data_aprovacao")
                      .withColumnRenamed("order_delivered_carrier_date", "data_entrega_transportada")
                      .withColumnRenamed("order_delivered_customer_date", "data_entrega_consumidor")
                      .withColumnRenamed("order_estimated_delivery_date", "data_estimada_entrega")) #renomeia os nomes das colunas de acordo com mapeamento

fat_pedidos = (fat_pedidos.withColumn("status", 
                    F.when(F.col("order_status") == "delivered", "entregue")
                    .when(F.col("order_status") == "canceled", "cancelado")
                    .when(F.col("order_status") == "shipped", "enviado")
                    .when(F.col("order_status") == "processing", "processado")
                    .when(F.col("order_status") == "invoiced", "faturado")
                    .when(F.col("order_status") == "unavailable", "indisponível")
                    .when(F.col("order_status") == "created", "criado")
                    .when(F.col("order_status") == "approved", "aprovado")
                    .otherwise(F.col("order_status"))).drop("order_status")) #aplicando a regra de negócio, criando a coluna status e aplicando a regra de negocio

fat_pedidos = fat_pedidos.withColumn(
    "tempo_entrega_dias",
    F.datediff(F.col("data_entrega_consumidor"), F.col("data_compra"))
)

fat_pedidos = fat_pedidos.withColumn(
    "tempo_entrega_estimado_dias",
    F.datediff(F.col("data_estimada_entrega"), F.col("data_compra"))#datediff calculca a diferenca entre os dias
)

fat_pedidos = fat_pedidos.withColumn(
    "diferenca_entrega_dias",
    F.col("tempo_entrega_dias") - F.col("tempo_entrega_estimado_dias")
)

fat_pedidos = fat_pedidos.withColumn(
    "entrega_no_prazo",
    F.when(
        (F.col("status") == "entregue") & (F.col("diferenca_entrega_dias") <= 0), "Sim"
    ).when(
        (F.col("status") == "entregue") & (F.col("diferenca_entrega_dias") > 0),  "Não"
    ).otherwise("Não Entregue")
) 

fat_pedidos = (fat_pedidos.withColumn("data_compra", F.col("data_compra").cast("date"))
               .withColumn("data_aprovacao", F.col("data_aprovacao").cast("date"))
               .withColumn("data_entrega_transportada", F.col("data_entrega_transportada").cast("date"))
               .withColumn("data_entrega_consumidor", F.col("data_entrega_consumidor").cast("date"))
               .withColumn("data_estimada_entrega", F.col("data_estimada_entrega").cast("date")))
#se for 0 ou negativo a entrega chegou no prazo, se for maior que 0 a entrega atrasou (faz a diferenca entre o tempo entrega e o tempo estimado e armazena esse valor em "diferenca_entrega_dias")

(fat_pedidos
 .write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(f"{catalogo}.{silver_db_name}.fat_pedidos")) #salva o data frame na camada silver



In [0]:
fat_pedidos.display()

In [0]:
fat_itens_pedidos = (df_order_items.withColumnRenamed("order_id", "id_pedido")
                      .withColumnRenamed("order_item_id", "id_item")
                      .withColumnRenamed("shipping_limit_date", "data_limite_envio")
                      .withColumnRenamed("product_id", "id_produto")
                      .withColumnRenamed("seller_id", "id_vendedor")
                      .withColumnRenamed("price", "preco_BRL")
                      .withColumnRenamed("freight_value", "preco_frete"))

fat_itens_pedidos = (fat_itens_pedidos.withColumn("data_limite_envio", (F.col("data_limite_envio")).cast("date")))

(fat_itens_pedidos
 .write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(f"{catalogo}.{silver_db_name}.fat_itens_pedidos"))



In [0]:
fat_itens_pedidos.display()

In [0]:
fat_pagamentos_pedidos = (df_order_payments.withColumnRenamed("order_id", "id_pedido")
                                .withColumnRenamed("payment_sequential", "sequencia_pagamento")
                                .withColumnRenamed("payment_value", "valor_pagamento")
                                .withColumnRenamed("payment_type", "tipo_pagamento")
                                .withColumnRenamed("payment_installments", "parcelas"))

fat_pagamentos_pedidos = (fat_pagamentos_pedidos.withColumn("tipo_pagamento", 
                                F.when(F.col("tipo_pagamento") == "credit_card", "Cartão de Crédito") 
                                .when(F.col("tipo_pagamento") == "boleto", "Boleto")
                                .when(F.col("tipo_pagamento") == "debit_card", "Cartão de Débito")
                                .when(F.col("tipo_pagamento") == "voucher", "Voucher")
                                .when(F.col("tipo_pagamento") == "not_defined", "Não Definido")
                                .otherwise(F.col("tipo_pagamento"))))
                                #cria uma nova coluna de tipo de pagamento pega os "tipos" da coluna payments_types e exclui a coluna pyment_type
(fat_pagamentos_pedidos.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(f"{catalogo}.{silver_db_name}.fat_pagamentos_pedidos"))


In [0]:
fat_pagamentos_pedidos.display()

In [0]:
fat_avaliacoes_pedidos = (df_order_reviews.withColumnRenamed("order_id", "id_pedido")
                                .withColumnRenamed("review_id", "id_avaliacao" )
                                .withColumnRenamed("review_score", "nota_avaliacao")
                                .withColumnRenamed("review_comment_title", "titulo_avaliacao_comentario")
                                .withColumnRenamed("review_comment_message", "mensagem_avaliacao_comentario")
                                .withColumnRenamed("review_creation_date", "data_criacao_avaliacao")
                                .withColumnRenamed("review_answer_timestamp", "data_resposta_avaliacao"))

fat_avaliacoes_pedidos = (fat_avaliacoes_pedidos.withColumn("data_criacao_avaliacao", F.try_to_timestamp(F.col("data_criacao_avaliacao")))
                                 .withColumn("data_resposta_avaliacao", F.try_to_timestamp(F.col("data_resposta_avaliacao"))))

fat_avaliacoes_pedidos = (fat_avaliacoes_pedidos.filter(F.col("id_pedido").isNotNull()) #filtra os dados onde o id_pedido é nulo
                                 .filter((F.col("data_criacao_avaliacao").isNotNull()) | (F.col("data_resposta_avaliacao") <= F.current_timestamp()))) #filtra os dados onde a data de criação da avaliação é nula ou a data de resposta da avaliação é menor que a data atual

fat_avaliacoes_pedidos = (fat_avaliacoes_pedidos.withColumn("titulo_avaliacao_comentario",
                                F.coalesce(F.col("titulo_avaliacao_comentario"), F.lit("Sem título"))).withColumn("mensagem_avaliacao_comentario",
                                F.coalesce(F.col("mensagem_avaliacao_comentario"), F.lit("Sem comentário")))) #F.lit cria uma coluna literal com o valor passado como argumento nesse caso o texto
                                #F.coalesce pega o primeiro valor não nulo da lista de colunas passadas
                                #ao repetir o mesmo nome das colunas o pyspark reescreve a coluna com a mesma regra

fat_avaliacoes_pedidos = (fat_avaliacoes_pedidos.withColumn("data_criacao_avaliacao", F.col("data_criacao_avaliacao").cast("date"))
                          .withColumn("data_resposta_avaliacao", F.col("data_resposta_avaliacao").cast("date")))

(fat_pagamentos_pedidos.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(f"{catalogo}.{silver_db_name}.fat_avaliacoes_pedidos"))

In [0]:
fat_avaliacoes_pedidos.display()

In [0]:
dim_produtos = (df_products.withColumnRenamed("product_id", "id_produto")
                        .withColumnRenamed("product_name", "nome_produto")
                        .withColumnRenamed("product_category_name", "categoria_produto")
                        .withColumnRenamed("product_name_lenght", "tamanho_nome_produto")
                        .withColumnRenamed("product_description_lenght", "tamanho_descricao_produto")
                        .withColumnRenamed("product_photos_qty", "quantidade_fotos")
                        .withColumnRenamed("product_weight_g", "peso_produto_gramas")
                        .withColumnRenamed("product_length_cm", "comprimento_centimetros")
                        .withColumnRenamed("product_height_cm", "altura_centimetros")
                        .withColumnRenamed("product_width_cm", "largura_centimetros"))

janela_deduplicacao_produtos = Window.partitionBy("id_produto").orderBy("timestamps_ingestion") #Remove os ID duplos de acordo com ordem decrescente de ingestao(recente)

(dim_produtos.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(f"{catalogo}.{silver_db_name}.dim_produtos"))

In [0]:
dim_produtos.display()

In [0]:
dim_vendedores = (df_sellers.withColumnRenamed("seller_id", "id_vendedor")
                  .withColumnRenamed("seller_name", "nome_vendedor")
                  .withColumnRenamed("seller_zip_code_prefix", "prefixo_cep")
                  .withColumnRenamed("seller_city", "cidade_vendedor")
                  .withColumnRenamed("seller_state", "estado_vendedor")
                  .withColumnRenamed("seller_registration_date", "data_registro_vendedor"))
                  
janela_deduplicacao_vendedores = Window.partitionBy("id_vendedor").orderBy("timestamps_ingestion")

dim_vendedores = (dim_vendedores.withColumn("cidade_vendedor", F.upper(F.col("cidade_vendedor"))) 
                  .withColumn("estado_vendedor", F.upper(F.col("estado_vendedor")))) #quando a coluna nova recebe o mesmo nome de uma existente o pyspark reescreve a coluna com a mesma regra

dim_vendedores = (dim_vendedores.withColumn("prefixo_cep", F.col("prefixo_cep").cast("string"))) #deixar cep como string pois se tiver 0 a esquerda como inteiro é ignorado
(dim_vendedores.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(f"{catalogo}.{silver_db_name}.dim_vendedores"))


In [0]:
dim_vendedores.display()

In [0]:
dim_categoria_produtos_traducao = (df_product_category_name_translation.withColumnRenamed("product_category_name", "nome_produto_pt")
                                    .withColumnRenamed("product_category_name_english", "nome_produto_en"))

(dim_categoria_produtos_traducao.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{silver_db_name}.dim_categoria_produtos_traducao"))


In [0]:
dim_categoria_produtos_traducao.display()

In [0]:
df_cotacao_raw = (df_cotacao
.withColumn("data_cotacao", F.to_date(F.col("dataHoraCotacao")))
.withColumnRenamed("cotacaoCompra", "cotacao_compra_usd")
.select("data_cotacao", "cotacao_compra_usd")
.groupBy("data_cotacao")
.agg(F.last("cotacao_compra_usd", ignorenulls=True).alias("cotacao_compra_usd")))

data_min = df_cotacao_raw.agg(F.min("data_cotacao")).collect()[0][0]
data_max = df_cotacao_raw.agg(F.max("data_cotacao")).collect()[0][0] #descobre a maior e a menor data para fazer um range de datas

df_calendario = spark.sql(f"""
SELECT explode(sequence(DATE '{data_min}', DATE '{data_max}', INTERVAL 1 DAY))  AS data_cotacao
""") #cria um data frame que pega as datas com intervalo de min e max do dataset

df_cotacao_completa = df_calendario.join(df_cotacao_raw, on="data_cotacao", how="left") #junta o df_calendario com o df_cotacao_raw

janela_cotacao = (Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, Window.currentRow)) #preenche os valores vazios (fim de semana) com ultimo valor existente

dim_cotacao_dolar = (df_cotacao_completa.withColumn("cotacao_compra_usd", F.last("cotacao_compra_usd", ignorenulls=True).over(janela_cotacao))
                     .withColumnRenamed("data_cotacao", "data")) #o .over faz com que a funcao normal vire uma Window fuction e nao permite que F.last quebre o df inteiro

(dim_cotacao_dolar.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{silver_db_name}.dim_cotacao_dolar"))


In [0]:
dim_cotacao_dolar.display()

In [0]:
df_pagamentos_agg = (spark.table("medalhao.silver.fat_pagamentos_pedidos").groupBy("id_pedido").agg(F.round(F.sum("valor_pagamento"), 2).alias("valor_total_pago_brl"))) #deixa os IDs em uma unica linha com isso resulta em uma tabela com duas colunas um com o id e outra com o valor total pago

df_pedidos = (spark.table("medalhao.silver.fat_pedidos").select("id_pedido", "id_consumidor", "status", "data_compra")) #salva apenas essas tabelas na tabela final

df_cotacao = (spark.table("medalhao.silver.dim_cotacao_dolar").withColumnRenamed("data", "data_compra_cotacao")) #renomia a tabela para o join nao gerar ambiguidade

fat_pedido_total = df_pedidos.join(df_pagamentos_agg, on="id_pedido", how="left") #leftJoin entre pedidos e pagmantos e garante q todos os pedidos sejam mantidos

fat_pedido_total = (fat_pedido_total.withColumn("data_compra_data", F.to_date(F.col("data_compra")))) #tranforma a data de compra para data ao inves de timestamp para o JOIN funcionar

fat_pedido_total = fat_pedido_total.join(df_cotacao.withColumnRenamed("data_compra_cotacao", "data_compra_data"), on="data_compra_data", how="left") #renomeia a coluna e faz um left join entre as tabelas de pedidos e cotacao

fat_pedido_total = (fat_pedido_total.withColumn("valor_total_pago_usd", F.round(F.col("valor_total_pago_brl") / F.col("cotacao_compra_usd"), 2))
                            .select("id_pedido", "id_consumidor", "status", 
                            F.round("valor_total_pago_brl", 2).alias("valor_total_pago_brl"),
                            F.round("valor_total_pago_usd", 2).alias("valor_total_pago_usd"),
                            F.col("data_compra").alias("data_pedido"))) #divide o valor total pago por cotacao para obter o valor total pago em USD e retorna a tabela final com as colunas desejadas e com duas casas decimais no valores

(fat_pedido_total.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{silver_db_name}.fat_pedido_total"))

In [0]:
fat_pedido_total.display()

In [0]:
tabelas_para_otimizar = [
("silver.fat_pedido_total", "id_pedido, data_pedido"),
("silver.fat_pedidos", "id_pedido, data_compra"),
("silver.fat_itens_pedidos", "id_pedido"),
("silver.fat_pagamentos_pedidos","id_pedido"),
]

for tabela, colunas in tabelas_para_otimizar:
    spark.sql(f"OPTIMIZE {catalogo}.{tabela} ZORDER BY ({colunas})")